# GuardianEye — Kaggle Orchestration Notebook

This notebook is a **thin orchestration wrapper only**.
All project logic lives in the repository's Python modules.

**Steps performed:**
1. Authenticate with GitHub (private repository)
2. Clone the repository
3. Install declared dependencies
4. Configure dataset and results paths
5. Run `make` targets

> **Security note:** Store your GitHub PAT in Kaggle Secrets
> (Account → Settings → Secrets → Add) as `GITHUB_TOKEN`.
> Enable it for this notebook session.  Never paste it into a cell.

## 0. Configuration — edit before running

In [ ]:
# ---------------------------------------------------------------------------
# User-configurable settings — adjust before running
# ---------------------------------------------------------------------------
REPO_OWNER = "your-github-username"   # e.g. "jdoe"
REPO_NAME  = "guardian-eye"
BRANCH     = "main"

# Kaggle input dataset path (set by Kaggle when datasets are attached)
# Each dataset archive should be uploaded as a separate Kaggle Dataset.
# Kaggle mounts them under /kaggle/input/<dataset-slug>/
KAGGLE_DATA_ROOT    = "/kaggle/input"          # adjust to your dataset slug
KAGGLE_RESULTS_ROOT = "/kaggle/working/results"

## 1. Load GitHub token from Kaggle Secrets

Add `GITHUB_TOKEN` via Account → Settings → Secrets.
Enable it for this session in the notebook settings panel.

In [ ]:
import os

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
if not GITHUB_TOKEN:
    raise RuntimeError(
        "GITHUB_TOKEN not found in environment.\n"
        "Add it as a Kaggle Secret and enable it for this session."
    )
print("GitHub token loaded (not displayed).")

## 2. Clone the repository

In [ ]:
import os
import subprocess

REPO_URL  = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
CLONE_DIR = f"/kaggle/working/{REPO_NAME}"

if not os.path.isdir(CLONE_DIR):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, CLONE_DIR],
        check=True,
    )
    print(f"Repository cloned to {CLONE_DIR}")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "origin", BRANCH], check=True)
    print(f"Repository updated at {CLONE_DIR}")

os.chdir(CLONE_DIR)
print(f"Working directory: {os.getcwd()}")

## 3. Install dependencies

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "-e", ".[dev]"], check=True)
print("Dependencies installed.")

## 4. Configure paths

Datasets must be uploaded as Kaggle Datasets (from archives produced by
`make pack-data`) and attached to this notebook before running.

In [ ]:
import os

os.makedirs(KAGGLE_RESULTS_ROOT, exist_ok=True)
os.environ["DATA_ROOT"]    = KAGGLE_DATA_ROOT
os.environ["RESULTS_ROOT"] = KAGGLE_RESULTS_ROOT

print(f"DATA_ROOT    = {KAGGLE_DATA_ROOT}")
print(f"RESULTS_ROOT = {KAGGLE_RESULTS_ROOT}")

## 5. Verify environment

In [ ]:
import subprocess
subprocess.run(["make", "data-check"], check=True)

## 6. Run the smoke test

In [ ]:
import subprocess
subprocess.run(["make", "smoke"], check=True)

## 7. Run tests and lint

In [ ]:
import subprocess
subprocess.run(["make", "test"], check=True)
subprocess.run(["make", "lint"], check=True)

## 8. Phase 1+ training targets

Uncomment the relevant target when the corresponding phase is ready.

In [ ]:
import subprocess

# subprocess.run(["make", "train-image"], check=True)
# subprocess.run(["make", "train-audio"], check=True)
# subprocess.run(["make", "pairs"], check=True)
# subprocess.run(["make", "fuse"], check=True)
# subprocess.run(["make", "evaluate"], check=True)
# subprocess.run(["make", "bda"], check=True)

print("Training targets are available in Phase 1+. See Makefile for details.")